# CLIMA-POLL — Pollination-success forecasting and chance-constrained drone scheduling

This notebook reproduces every experiment, table value and figure of the manuscript *"Physiologically Constrained Pollination-Success Forecasting and Scenario-Based Chance-Constrained Drone Scheduling under Climate Variability"*.

**How to run (Google Colab, free tier):** `Runtime → Run all`. Only preinstalled packages are used (NumPy, SciPy ≥ 1.9 with the HiGHS MILP solver, scikit-learn, Matplotlib). A CPU runtime is sufficient.

* Full run: 5 replicates of 6 test seasons each, followed by the budget, wind-reliability, frontier and ensemble-size studies (roughly 2–3 h on the free CPU runtime in total).
* Quick check: set `os.environ["FAST"] = "1"` in the first code cell (2 replicates, smaller studies).

Outputs are written to `results/` (JSON) and `figs/` (PNG, 400 dpi, grayscale). `numbers.json` holds every number quoted in the paper. The experiments are simulation-based; no field data are used.


# Microclimate-aware pollination-window forecasting and chance-constrained drone pollination scheduling
Reproduces all numbers, tables and figures of the manuscript. Runs on the free Colab tier (CPU is sufficient).
Set FAST=1 in the environment for a quick smoke test.

In [ ]:
import os, json, time, math, warnings
# os.environ['FAST'] = '1'   # uncomment for a quick smoke test
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds, minimize
from scipy.sparse import lil_matrix
from scipy.stats import wilcoxon
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
FAST = os.environ.get("FAST", "0") == "1"
OUT = os.environ.get("OUT", "results"); os.makedirs(OUT, exist_ok=True)
N_REP = 2 if FAST else 5
DAYS = 14; HOURS = np.arange(6, 20)          # daylight operating hours 06:00-19:00
NH = len(HOURS); NB = 8                        # blocks per orchard
M_ENS = 16; LEAD_DAYS = 2                      # ensemble members, planning horizon (days)

## 1. Stochastic weather generator (station) with heatwaves and cold spells

In [ ]:
def esat(T):           # saturation vapour pressure, kPa (Tetens)
    return 0.6108 * np.exp(17.27 * T / (T + 237.3))

def weather_season(rng, regime="nominal"):
    """Hourly station weather for DAYS days (24 h each)."""
    H24 = 24 * DAYS; hod = np.arange(H24) % 24; day = np.arange(H24) // 24
    base = 13.0 + 0.18 * np.arange(DAYS)                      # spring warming trend
    anom = np.zeros(DAYS); a = rng.normal(0, 1.5)
    for d in range(DAYS): a = 0.7 * a + rng.normal(0, 1.6); anom[d] = a
    cloud = np.zeros(DAYS); c = rng.uniform(0.2, 0.7)
    for d in range(DAYS): c = np.clip(0.6 * c + 0.4 * rng.beta(1.6, 1.8), 0, 1); cloud[d] = c
    event = np.zeros(DAYS)
    if regime in ("heatwave", "cold_spell"):
        s = rng.integers(3, DAYS - 5); L = rng.integers(3, 6)
        mag = rng.uniform(10.0, 13.0) if regime == "heatwave" else -rng.uniform(8.0, 10.0)
        prof = np.sin(np.linspace(0.2, np.pi - 0.2, L))
        event[s:s + L] = mag * prof[: len(event[s:s + L])]
    Tmean = base + anom + event
    amp = 7.5 * (1 - 0.6 * cloud) + (1.5 if regime == "heatwave" else 0) * (event > 0)
    T = Tmean[day] + amp[day] * np.sin(2 * np.pi * (hod - 9) / 24)
    e = 0.0
    noise = np.zeros(H24)
    for h in range(H24): e = 0.8 * e + rng.normal(0, 0.45); noise[h] = e
    T = T + noise
    spread = np.clip(5.0 + 2.5 * rng.standard_normal(DAYS) * 0.6 - 3.5 * cloud + (2.5 if regime == "heatwave" else 0) * (event > 1), 1.0, 14.0)
    Td = Tmean[day] - spread[day] + 0.8 * rng.standard_normal(H24) * 0.5
    Td = np.minimum(Td, T - 0.2)
    RH = 100 * esat(Td) / esat(T)
    wmean = rng.weibull(2.0, DAYS) * 2.9 + 0.7
    w = wmean[day] * (1 + 0.45 * np.sin(2 * np.pi * (hod - 10) / 24))
    ew = 0.0; wn = np.zeros(H24)
    for h in range(H24): ew = 0.7 * ew + rng.normal(0, 0.45); wn[h] = ew
    wind = np.clip(w + wn, 0.1, None)
    solar = np.clip(np.sin(np.pi * (hod - 6) / 13), 0, None) * 900 * (1 - 0.7 * cloud[day])
    rain = (rng.random(H24) < 0.12 * (cloud[day] > 0.65)) & (hod > 5)
    for h in np.where(rain)[0][:]:                                 # rain episodes last a few hours
        rain[h:h + rng.integers(1, 4)] = True
    return dict(T=T, RH=np.clip(RH, 8, 100), wind=wind, solar=solar, rain=rain.astype(float), Tmean=Tmean, regime=regime, event=event)

## 2. Orchard microclimate, phenology and pollen physiology (ground-truth simulator)

In [ ]:
def vpd(T, RH): return esat(T) * (1 - RH / 100.0)

def true_success(T, RH, wind, solar, rain, cult):
    """Probability that pollen applied in this hour germinates on a receptive stigma (truth, not known to models)."""
    Tmin, Topt, Tmax = cult["Tmin"], cult["Topt"], cult["Tmax"]
    Tc = np.clip(T, Tmin + 1e-3, Tmax - 1e-3)
    gT = ((Tmax - Tc) / (Tmax - Topt)) * ((Tc - Tmin) / (Topt - Tmin)) ** ((Topt - Tmin) / (Tmax - Topt))
    gT = np.where((T <= Tmin) | (T >= Tmax), 0.0, gT)
    D = vpd(T, RH)
    gV = np.exp(-np.maximum(0, D - 0.9) * (0.55 + 0.03 * np.maximum(T - 20, 0)))   # desiccation stronger when hot
    dep = 1.0 / (1.0 + np.exp((wind - 4.2) / 0.55))                                  # spray deposition vs wind
    uv = 1 - 0.12 * np.clip(solar / 900, 0, 1)                                       # UV damage (unmodelled by forecaster)
    return np.clip(0.92 * gT * gV * dep * uv * (1 - rain), 0, 1)

def orchard(rng, regime):
    st = weather_season(rng, regime)
    off_T = rng.normal(0, 0.9, NB); expo = rng.uniform(0.7, 1.25, NB); off_RH = rng.normal(0, 4, NB)
    cult = dict(Tmin=rng.uniform(3, 6), Topt=rng.uniform(20, 23), Tmax=rng.uniform(33, 36))
    blk = {}
    for k in ["T", "RH", "wind", "solar", "rain"]:
        if k == "T": blk[k] = st["T"][None] + off_T[:, None] + 0.4 * rng.standard_normal((NB, st["T"].size)) * 0.3
        elif k == "RH": blk[k] = np.clip(st["RH"][None] + off_RH[:, None], 8, 100)
        elif k == "wind": blk[k] = st["wind"][None] * expo[:, None]
        else: blk[k] = np.repeat(st[k][None], NB, 0)
    blk["succ"] = true_success(blk["T"], blk["RH"], blk["wind"], blk["solar"], blk["rain"], cult)
    # phenology: thermal-time driven anthesis; receptive window shortens with temperature
    n_fl = rng.integers(900, 1300, NB)
    gdh = np.cumsum(np.maximum(blk["T"] - 4.5, 0), axis=1)                           # growing degree hours
    thr = np.array([rng.normal(1500 + 220 * b / NB, 280, n) for b, n in enumerate(n_fl)], dtype=object)
    return dict(st=st, blk=blk, off_T=off_T, expo=expo, off_RH=off_RH, cult=cult, n_fl=n_fl, gdh=gdh, thr=thr, regime=regime)

class Phenology:
    """Flower-level receptivity clock per block; tracks fertilisation state."""
    def __init__(s, O, rng):
        s.O = O; s.open_h = []; s.end_h = []; s.fert = []
        Tm = O["blk"]["T"]
        for b in range(NB):
            thr = np.sort(O["thr"][b]); gd = O["gdh"][b]
            oh = np.searchsorted(gd, thr).astype(float)                              # hour of anthesis
            oh = np.minimum(oh, gd.size + 100)
            # receptive window (hours): 60 h at 15 C mean, shorter when warm
            Tw = np.array([Tm[b, int(min(h, gd.size - 1)):int(min(h + 48, gd.size))].mean() if h < gd.size else 15 for h in oh])
            L = 60.0 * np.exp(-0.055 * (Tw - 15.0)) * rng.uniform(0.85, 1.15, oh.size)
            s.open_h.append(oh + 6.0); s.end_h.append(oh + 6.0 + L); s.fert.append(np.zeros(oh.size, bool))
    def pool(s, b, h):
        m = (s.open_h[b] <= h) & (s.end_h[b] > h) & ~s.fert[b]; return m
    def count(s, b, h): return int(s.pool(b, h).sum())

## 3. Forecast products: station ensemble forecasts and in-orchard sensor history

In [ ]:
LEAD_SD = dict(T=(0.8, 0.045), RH=(4.0, 0.16), wind=(0.55, 0.02))

def ensemble_forecast(O, day, rng, m=M_ENS):
    """Station ensemble for hours [day*24, (day+LEAD_DAYS)*24). Returns dict var -> (m, H) arrays (station level)."""
    st = O["st"]; h0 = day * 24; H = LEAD_DAYS * 24; h1 = min(h0 + H, 24 * DAYS); Hh = h1 - h0
    out = {}
    lead = np.arange(Hh)
    for v in ["T", "RH", "wind"]:
        a, b = LEAD_SD[v]; sd = a + b * lead
        # systematic forecast error shared by members (unknown to the user) + member spread
        sys_err = np.zeros(Hh); e = rng.normal(0, 1)
        for k in range(Hh): e = 0.97 * e + rng.normal(0, 0.25); sys_err[k] = e
        centre = st[v][h0:h1] + 0.8 * sd * sys_err
        mem = np.zeros((m, Hh))
        for j in range(m):
            e = rng.normal(0, 1); path = np.zeros(Hh)
            for k in range(Hh): e = 0.95 * e + rng.normal(0, 0.31); path[k] = e
            mem[j] = centre + 0.9 * sd * path
        if v == "RH": mem = np.clip(mem, 8, 100)
        if v == "wind": mem = np.clip(mem, 0.1, None)
        out[v] = mem
    out["solar"] = np.repeat(st["solar"][h0:h1][None], m, 0) * rng.uniform(0.85, 1.15, (m, 1))
    p_rain = np.clip(st["rain"][h0:h1] * 0.7 + 0.08, 0, 1)
    out["rain"] = (rng.random((m, Hh)) < p_rain).astype(float)
    out["h0"] = h0; out["H"] = Hh
    return out

def sensor_offsets(O, day, rng, window=72, use=True):
    """Block offsets estimated from in-orchard sensors (block minus station) over the last `window` hours."""
    if not use or day == 0:
        return np.zeros(NB), np.ones(NB), np.zeros(NB)
    h1 = day * 24; h0 = max(0, h1 - window)
    bT = O["blk"]["T"][:, h0:h1] + rng.normal(0, 0.3, (NB, h1 - h0)); sT = O["st"]["T"][h0:h1]
    bW = O["blk"]["wind"][:, h0:h1] * rng.uniform(0.95, 1.05, (NB, 1)); sW = O["st"]["wind"][h0:h1]
    bR = O["blk"]["RH"][:, h0:h1] + rng.normal(0, 2, (NB, h1 - h0)); sR = O["st"]["RH"][h0:h1]
    oT = (bT - sT).mean(1); ex = (bW * sW).sum(1) / (sW * sW).sum(); oR = (bR - sR).mean(1)
    return oT, ex, oR

## 4. Pollination-success forecasters
* PCF (proposed): physiologically constrained factorised model; cardinal-temperature beta x VPD desiccation x wind deposition x rain.
* Baselines: hour-of-day climatology, gradient-boosted trees, MLP on the same inputs.

In [ ]:
def feats(T, RH, W, S, Rn, hod):
    T, RH, W, S, Rn, hod = np.broadcast_arrays(T, RH, W, S, Rn, hod)
    return np.stack([T, RH, vpd(T, RH), W, S / 900, Rn, np.sin(2 * np.pi * hod / 24), np.cos(2 * np.pi * hod / 24)], -1)

class PCF:
    """theta = (Tmin, Topt, Tmax, a_v, d0, w50, s_w, p_max)."""
    def __init__(s): s.th = np.array([4.0, 21.0, 34.0, 0.6, 1.0, 4.0, 0.6, 0.9])
    @staticmethod
    def f(th, T, RH, W, Rn):
        Tmin, Topt, Tmax, av, d0, w50, sw, pm = th
        Tmin = min(Tmin, Topt - 1); Tmax = max(Tmax, Topt + 1)
        Tc = np.clip(T, Tmin + 1e-3, Tmax - 1e-3)
        gT = ((Tmax - Tc) / (Tmax - Topt)) * ((Tc - Tmin) / (Topt - Tmin)) ** ((Topt - Tmin) / (Tmax - Topt))
        gT = np.where((T <= Tmin) | (T >= Tmax), 0.0, gT)
        gV = np.exp(-max(av, 0) * np.maximum(0, vpd(T, RH) - d0))
        dep = 1 / (1 + np.exp((W - w50) / max(sw, 0.05)))
        return np.clip(1 / (1 + np.exp(-pm)) * gT * gV * dep * (1 - Rn), 1e-5, 1 - 1e-5)
    def fit(s, T, RH, W, Rn, k, n):
        def nll(th):
            p = s.f(th, T, RH, W, Rn); return -np.sum(k * np.log(p) + (n - k) * np.log(1 - p)) / n.sum()
        best = None
        for init in [s.th, s.th + np.array([1, -1, 1, 0.2, 0.2, -0.3, 0.1, 0.3])]:
            r = minimize(nll, init, method="Nelder-Mead", options=dict(maxiter=4000, xatol=1e-3, fatol=1e-7))
            if best is None or r.fun < best.fun: best = r
        s.th = best.x; return s
    def predict(s, T, RH, W, S, Rn, hod): T, RH, W, Rn = np.broadcast_arrays(T, RH, W, Rn); return s.f(s.th, T, RH, W, Rn)

class Clim:
    def fit(s, hod, k, n):
        s.p = np.array([k[hod == h].sum() / max(1, n[hod == h].sum()) for h in range(24)]); return s
    def predict(s, T, RH, W, S, Rn, hod): T, hod = np.broadcast_arrays(T, hod); return np.clip(s.p[hod.astype(int)], 1e-4, 1 - 1e-4)

class GBT:
    def fit(s, X, k, n):
        Xr = np.repeat(X, 2, 0); y = np.tile([1, 0], len(X)); w = np.stack([k, n - k], 1).ravel()
        s.m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_leaf_nodes=31, random_state=0).fit(Xr, y, sample_weight=w); return s
    def predict(s, T, RH, W, S, Rn, hod): return np.clip(s.m.predict_proba(feats(T, RH, W, S, Rn, hod).reshape(-1, 8))[:, 1].reshape(np.broadcast_shapes(np.shape(T), np.shape(RH), np.shape(W), np.shape(S))), 1e-4, 1 - 1e-4)

class MLP:
    def fit(s, X, k, n, seed=0):
        s.sc = StandardScaler().fit(X)
        Xr = np.repeat(s.sc.transform(X), 2, 0); y = np.tile([1, 0], len(X)); w = np.stack([k, n - k], 1).ravel()
        idx = np.random.default_rng(seed).choice(len(Xr), size=min(len(Xr), 60000), p=w / w.sum())
        s.m = MLPClassifier((64, 64), max_iter=300, random_state=seed, early_stopping=True).fit(Xr[idx], y[idx]); return s
    def predict(s, T, RH, W, S, Rn, hod):
        return np.clip(s.m.predict_proba(s.sc.transform(feats(T, RH, W, S, Rn, hod).reshape(-1, 8)))[:, 1].reshape(np.broadcast_shapes(np.shape(T), np.shape(RH), np.shape(W), np.shape(S))), 1e-4, 1 - 1e-4)

def trial_data(orchards, rng, per_hour=2, grains=25):
    """Controlled pollen-viability trials: hourly germination counts on detached stigmas in each block."""
    T, RH, W, S, Rn, hod, K, Nn = [], [], [], [], [], [], [], []
    for O in orchards:
        B = O["blk"]; H = B["T"].shape[1]
        for b in range(NB):
            hh = np.array([h for h in range(H) if 6 <= h % 24 < 20])
            for _ in range(per_hour):
                p = B["succ"][b, hh]; k = rng.binomial(grains, p)
                T.append(B["T"][b, hh] + rng.normal(0, 0.3, hh.size)); RH.append(B["RH"][b, hh]); W.append(B["wind"][b, hh])
                S.append(B["solar"][b, hh]); Rn.append(B["rain"][b, hh]); hod.append(hh % 24); K.append(k); Nn.append(np.full(hh.size, grains))
    c = lambda L: np.concatenate(L)
    return c(T), c(RH), c(W), c(S), c(Rn), c(hod), c(K).astype(float), c(Nn).astype(float)

## 5. Mission scheduling
Expected fertilised flowers of block b on day d: R_bd * (1 - exp(-kappa * sum_h x_bh v_bh)), concave in sorties.
Scenario (ensemble) average, outer-approximated by tangent cuts; wind chance constraint by scenario counting.

In [ ]:
KAPPA = 0.35; X_MAX = 3; FLEET_PER_H = 4; SORTIES_PER_DAY = 24; POLLEN_G = 2.5; POLLEN_PER_DAY = 40.0
W_MAX = 4.5; EPS = 0.10
TANG = np.array([0.0, 0.2, 0.45, 0.8, 1.2, 1.8, 2.6, 4.0])

def solve_schedule(V, R, wind_ok, days_in_h, eps_cost=1e-3):
    """V: (S, NB, Hp) success per scenario; R: (NB, Dp) receptive pools per plan day; wind_ok: (NB, Hp) bool;
    days_in_h: (Hp,) day index (0..Dp-1) of each plan hour. Returns integer sorties x (NB, Hp)."""
    S, nb, Hp = V.shape; Dp = R.shape[1]
    nx = nb * Hp; nt = S * nb * Dp; nvar = nx + nt
    ix = lambda b, h: b * Hp + h; it = lambda s, b, d: nx + (s * nb + b) * Dp + d
    c = np.zeros(nvar)
    for s in range(S):
        for b in range(nb):
            for d in range(Dp): c[it(s, b, d)] = -R[b, d] / S
    c[:nx] += eps_cost * POLLEN_G
    rows = []; lo = []; hi = []
    A = lil_matrix((nt * len(TANG) + Hp + Dp * 2, nvar))
    r = 0
    for s in range(S):
        for b in range(nb):
            for d in range(Dp):
                hs = np.where(days_in_h == d)[0]
                for z0 in TANG:          # t <= phi(z0) + phi'(z0) (z - z0), phi(z) = 1 - exp(-kappa z)
                    g = KAPPA * np.exp(-KAPPA * z0); A[r, it(s, b, d)] = 1.0
                    for h in hs: A[r, ix(b, h)] = -g * V[s, b, h]
                    lo.append(-np.inf); hi.append((1 - np.exp(-KAPPA * z0)) - g * z0); r += 1
    for h in range(Hp):
        for b in range(nb): A[r, ix(b, h)] = 1.0
        lo.append(0); hi.append(FLEET_PER_H); r += 1
    for d in range(Dp):
        hs = np.where(days_in_h == d)[0]
        for b in range(nb):
            for h in hs: A[r, ix(b, h)] = 1.0
        lo.append(0); hi.append(SORTIES_PER_DAY); r += 1
        for b in range(nb):
            for h in hs: A[r, ix(b, h)] = POLLEN_G
        lo.append(0); hi.append(POLLEN_PER_DAY); r += 1
    A = A[:r].tocsr()
    ub = np.concatenate([np.where(wind_ok.ravel(), X_MAX, 0), np.ones(nt)])
    integ = np.concatenate([np.ones(nx), np.zeros(nt)])
    res = milp(c, constraints=LinearConstraint(A, lo, hi), integrality=integ, bounds=Bounds(np.zeros(nvar), ub),
               options=dict(time_limit=20, mip_rel_gap=1e-3))
    if res.x is None: return np.zeros((nb, Hp), int)
    return np.round(res.x[:nx]).astype(int).reshape(nb, Hp)

## 6. Season simulation under a scheduling policy

In [ ]:
def daylight_idx(h0, H):
    return np.array([k for k in range(H) if 6 <= (h0 + k) % 24 < 20])

def run_season(O, policy, models, rng_seed, eps=EPS, use_sensors=True, scenarios=True, forecaster="pcf", margin=0.0, log=False, pollen_day=None):
    rng = np.random.default_rng(rng_seed)
    ph = Phenology(O, np.random.default_rng(rng_seed + 1))
    B = O["blk"]; total = int(sum(O["n_fl"]))
    sorties = 0; viol = 0; fert = 0; pollen = 0.0; solve_t = []; LOG = []
    global POLLEN_PER_DAY
    _pd_saved = POLLEN_PER_DAY
    if pollen_day is not None: POLLEN_PER_DAY = pollen_day
    for day in range(DAYS):
        F = ensemble_forecast(O, day, rng)
        h0, Hh = F["h0"], F["H"]; dl = daylight_idx(h0, Hh); hours = h0 + dl
        oT, ex, oR = sensor_offsets(O, day, rng, use=use_sensors)
        hod = hours % 24
        # block-level ensemble
        Tm = F["T"][:, None, dl] + oT[None, :, None]; Wm = F["wind"][:, None, dl] * ex[None, :, None]
        Rm = np.clip(F["RH"][:, None, dl] + oR[None, :, None], 8, 100); Sm = F["solar"][:, None, dl]; Rn = F["rain"][:, None, dl] * np.ones((1, NB, 1))
        days_in_h = dl // 24
        Dp = int(days_in_h.max()) + 1
        # receptive pools expected per plan day (flowers receptive at noon of that day, not yet fertilised)
        R = np.array([[ph.count(b, h0 + 24 * d + 12) for d in range(Dp)] for b in range(NB)], float)
        x = np.zeros((NB, dl.size), int)
        if policy == "calendar":                       # fixed daily round: 2 sorties per block from 08:00, fleet-limited
            for b in range(NB):
                hh = day * 24 + 8 + b // (FLEET_PER_H // 2)            # 2 blocks per hour, 08:00-11:00
                k = np.where(hours == hh)[0]
                if k.size: x[b, k[0]] = 2
        elif policy == "greedy":
            mT, mW, mR = Tm.mean(0), Wm.mean(0), Rm.mean(0)
            ok = (mT > 12) & (mT < 28) & (mW < W_MAX - 0.8) & (Rn.mean(0) < 0.3) & (vpd(mT, mR) < 1.8)
            used_day = 0; used_p = 0.0
            for k, h in enumerate(hours):
                if h // 24 != day: continue
                cap = FLEET_PER_H
                order = np.argsort(-R[:, 0])
                for b in order:
                    if cap <= 0 or used_day >= SORTIES_PER_DAY or used_p + POLLEN_G > POLLEN_PER_DAY: break
                    if ok[b, k] and R[b, 0] > 0 and x[b, :].sum() < 3:
                        n_ = min(2, cap); x[b, k] += n_; cap -= n_; used_day += n_; used_p += n_ * POLLEN_G
        else:
            mdl = models[forecaster]
            if policy == "milp_hybrid":                 # mean-forecast objective, scenario chance constraint
                mT, mW, mR = Tm.mean(0), Wm.mean(0), Rm.mean(0)
                V = mdl.predict(mT, mR, mW, Sm.mean(0), (Rn.mean(0) > 0.5).astype(float), np.broadcast_to(hod, mT.shape))[None]
                wind_ok = (Wm > W_MAX).mean(0) <= eps
                t0 = time.time(); x = solve_schedule(V, R, wind_ok, days_in_h); solve_t.append(time.time() - t0)
            if policy in ("milp_mean", "milp_scen", "milp_oracle"):
                if policy == "milp_oracle":
                    V = B["succ"][None, :, hours]; wind_ok = B["wind"][:, hours] <= W_MAX
                elif policy == "milp_mean" or not scenarios:
                    mT, mW, mR = Tm.mean(0), Wm.mean(0), Rm.mean(0)
                    V = mdl.predict(mT, mR, mW, Sm.mean(0), (Rn.mean(0) > 0.5).astype(float), np.broadcast_to(hod, mT.shape))[None]
                    wind_ok = mW <= W_MAX - margin
                else:
                    V = mdl.predict(Tm, Rm, Wm, Sm, Rn, np.broadcast_to(hod, Tm.shape))
                    wind_ok = (Wm > W_MAX).mean(0) <= eps
                t0 = time.time(); x = solve_schedule(V, R, wind_ok, days_in_h); solve_t.append(time.time() - t0)
            # execute only the first day of the plan
            x[:, days_in_h > 0] = 0
        # ----- execute with true weather -----
        for k, h in enumerate(hours):
            if h // 24 != day: continue
            for b in range(NB):
                n = int(x[b, k])
                if n == 0: continue
                sorties += n; pollen += n * POLLEN_G
                if log: LOG.append((int(h % 24), float(B["succ"][b, h]), float(B["T"][b, h]), float(B["wind"][b, h]), n))
                if B["wind"][b, h] > W_MAX: viol += n
                m = ph.pool(b, h); idx = np.where(m)[0]
                if idx.size == 0: continue
                p = 1 - np.exp(-KAPPA * n * B["succ"][b, h])
                hit = idx[rng.random(idx.size) < p]; ph.fert[b][hit] = True; fert += hit.size
        # background insect pollination (small, weather dependent)
        for h in range(day * 24 + 6, day * 24 + 20):
            for b in range(NB):
                idx = np.where(ph.pool(b, h))[0]
                if idx.size: hit = idx[rng.random(idx.size) < 0.004 * B["succ"][b, h]]; ph.fert[b][hit] = True
    nat = sum(int(f.sum()) for f in ph.fert) - fert
    POLLEN_PER_DAY = _pd_saved
    extra = dict(log=LOG) if log else {}
    return dict(**extra, fruit_set=sum(int(f.sum()) for f in ph.fert) / total, art_fert=fert, nat_fert=nat, sorties=sorties, viol=viol,
                viol_rate=viol / max(1, sorties), pollen=pollen, pollen_per_fruit=pollen / max(1, fert), solve_s=float(np.mean(solve_t)) if solve_t else 0.0)

## 7. Forecast evaluation helpers

In [ ]:
def brier(p, k, n): return float(np.sum(n * (p * p) - 2 * p * k + k) / n.sum())   # per-grain Brier score
def logloss(p, k, n): p = np.clip(p, 1e-5, 1 - 1e-5); return float(-np.sum(k * np.log(p) + (n - k) * np.log(1 - p)) / n.sum())
def mae_true(p, ptrue): return float(np.mean(np.abs(p - ptrue)))

def calib_curve(p, k, n, bins=10):
    e = np.linspace(0, 1, bins + 1); out = []
    for i in range(bins):
        m = (p >= e[i]) & (p < e[i + 1] + (i == bins - 1))
        if m.any(): out.append([float(p[m].mean()), float(k[m].sum() / n[m].sum()), int(m.sum())])
    return out

TRAIN_T = [None, None]
def forecast_skill(O, models, rng, use_sensors=True):
    """Day-ahead forecast skill against true block success probabilities and simulated trial outcomes."""
    P = {k: [] for k in models}; truth = []; Ks = []; Ns = []; lead = []; Tt = []
    for day in range(DAYS - 1):
        F = ensemble_forecast(O, day, rng); h0, Hh = F["h0"], F["H"]; dl = daylight_idx(h0, Hh); hours = h0 + dl
        oT, ex, oR = sensor_offsets(O, day, rng, use=use_sensors); hod = hours % 24
        Tm = F["T"][:, None, dl] + oT[None, :, None]; Wm = F["wind"][:, None, dl] * ex[None, :, None]
        Rm = np.clip(F["RH"][:, None, dl] + oR[None, :, None], 8, 100); Sm = F["solar"][:, None, dl]; Rn = F["rain"][:, None, dl] * np.ones((1, NB, 1))
        pt = O["blk"]["succ"][:, hours]
        for name, mdl in models.items():
            P[name].append(mdl.predict(Tm, Rm, Wm, Sm, Rn, np.broadcast_to(hod, Tm.shape)).mean(0))
        Tt.append(O["blk"]["T"][:, hours]); truth.append(pt); k = rng.binomial(25, pt); Ks.append(k); Ns.append(np.full(pt.shape, 25)); lead.append(np.broadcast_to(dl // 24, pt.shape))
    c = lambda L: np.concatenate([x.ravel() for x in L])
    t, K, Nn, ld, Ttrue = c(truth), c(Ks).astype(float), c(Ns).astype(float), c(lead), c(Tt)
    out = {}
    for name in models:
        p = c(P[name])
        oor = (Ttrue > TRAIN_T[1]) | (Ttrue < TRAIN_T[0]) if TRAIN_T[0] is not None else np.zeros_like(t, bool)
        out[name] = dict(brier=brier(p, K, Nn), logloss=logloss(p, K, Nn), mae=mae_true(p, t), mae_d1=mae_true(p[ld == 0], t[ld == 0]),
                         mae_d2=mae_true(p[ld == 1], t[ld == 1]), calib=calib_curve(p, K, Nn),
                         mae_oor=mae_true(p[oor], t[oor]) if oor.any() else None, n_oor=int(oor.sum()), brier_oor=brier(p[oor], K[oor], Nn[oor]) if oor.any() else None)
    return out

## 8. Replicate driver

In [ ]:
REGIMES_TEST = ["nominal", "nominal", "heatwave", "heatwave", "cold_spell", "cold_spell"]
POLICIES = ["calendar", "greedy", "milp_mean", "milp_scen", "milp_oracle"]

def replicate(rep, log=print):
    t0 = time.time(); rng = np.random.default_rng(500 + rep)
    train = [orchard(np.random.default_rng(1000 * rep + i), "nominal") for i in range(8)]
    test = [orchard(np.random.default_rng(1000 * rep + 100 + i), r) for i, r in enumerate(REGIMES_TEST)]
    T, RH, W, S, Rn, hod, K, Nn = trial_data(train, rng)
    X = feats(T, RH, W, S, Rn, hod); TRAIN_T[0], TRAIN_T[1] = float(np.percentile(T, 0.5)), float(np.percentile(T, 99.5))
    models = dict(pcf=PCF().fit(T, RH, W, Rn, K, Nn), clim=Clim().fit(hod, K, Nn), gbt=GBT().fit(X, K, Nn), mlp=MLP().fit(X, K, Nn, seed=rep))
    log(f"[rep {rep}] forecasters fitted in {time.time()-t0:.0f}s; trial records {len(K)}; PCF theta={np.round(models['pcf'].th,2)}")
    res = dict(rep=rep, theta=models["pcf"].th.tolist(), skill={}, seasons=[], ablation=[])
    # forecast skill per regime
    for i, O in enumerate(test):
        sk = forecast_skill(O, models, np.random.default_rng(7000 + 10 * rep + i))
        res["skill"].setdefault(O["regime"], []).append(sk)
    sk_nos = forecast_skill(test[0], {"pcf": models["pcf"]}, np.random.default_rng(7000 + 10 * rep), use_sensors=False)
    res["skill_no_sensor"] = sk_nos
    # policies
    for i, O in enumerate(test):
        for pol in POLICIES:
            r = run_season(O, pol, models, rng_seed=9000 + 100 * rep + i)
            r.update(policy=pol, regime=O["regime"], season=i); res["seasons"].append(r)
        # ablations on the proposed scheduler
        for tag, kw in [("gbt_forecaster", dict(forecaster="gbt")), ("mlp_forecaster", dict(forecaster="mlp")), ("no_sensors", dict(use_sensors=False)),
                        ("eps_0.00", dict(eps=0.0)), ("eps_0.20", dict(eps=0.2)), ("eps_0.35", dict(eps=0.35))]:
            r = run_season(O, "milp_scen", models, rng_seed=9000 + 100 * rep + i, **kw); r.update(tag=tag, regime=O["regime"], season=i); res["ablation"].append(r)
        log(f"[rep {rep}] season {i} ({O['regime']}) done, {time.time()-t0:.0f}s")
    # example season traces for figures (rep 0 only)
    if rep == 0:
        O = test[2]
        res["trace"] = dict(T=O["blk"]["T"][0].tolist(), wind=O["blk"]["wind"][0].tolist(), succ=O["blk"]["succ"][0].tolist(), RH=O["blk"]["RH"][0].tolist())
        grid = np.linspace(-2, 42, 89)
        res["curves"] = dict(T=grid.tolist(), pcf=models["pcf"].predict(grid, np.full_like(grid, 60), np.full_like(grid, 1.5), np.full_like(grid, 400), np.zeros_like(grid), np.full_like(grid, 12)).tolist(),
                             gbt=models["gbt"].predict(grid, np.full_like(grid, 60), np.full_like(grid, 1.5), np.full_like(grid, 400), np.zeros_like(grid), np.full_like(grid, 12)).tolist(),
                             mlp=models["mlp"].predict(grid, np.full_like(grid, 60), np.full_like(grid, 1.5), np.full_like(grid, 400), np.zeros_like(grid), np.full_like(grid, 12)).tolist(),
                             truth=true_success(grid, np.full_like(grid, 60), np.full_like(grid, 1.5), np.full_like(grid, 400), np.zeros_like(grid), test[0]["cult"]).tolist(),
                             train_T=[float(np.percentile(T, 1)), float(np.percentile(T, 99))])
    log(f"[rep {rep}] finished in {time.time()-t0:.0f}s")
    return res

if True:  # main experiment
    allres = []
    for rep in range(N_REP):
        allres.append(replicate(rep)); json.dump(allres, open(f"{OUT}/results.json", "w"))
    print("finished")

## 9. Risk-return frontier: fruit set versus wind-limit violations

In [ ]:
import json, numpy as np, time
import sys
C = sys.modules['__main__']
OUT = C.OUT; fr = []
for rep in range(2 if C.FAST else 3):
    rng = np.random.default_rng(500 + rep)
    train = [C.orchard(np.random.default_rng(1000 * rep + i), "nominal") for i in range(8)]
    test = [C.orchard(np.random.default_rng(1000 * rep + 100 + i), r) for i, r in enumerate(C.REGIMES_TEST)]
    T, RH, W, S, Rn, hod, K, Nn = C.trial_data(train, rng)
    models = dict(pcf=C.PCF().fit(T, RH, W, Rn, K, Nn))
    for i, O in enumerate(test):
        seed = 9000 + 100 * rep + i
        for m in [0.0, 0.5, 1.0, 1.5]:
            r = C.run_season(O, "milp_mean", models, seed, margin=m); r.update(rep=rep, season=i, regime=O["regime"], family="mean+margin", level=m); fr.append(r)
        for e in [0.05, 0.10, 0.20, 0.35, 0.50]:
            r = C.run_season(O, "milp_scen", models, seed, eps=e); r.update(rep=rep, season=i, regime=O["regime"], family="scenario", level=e); fr.append(r)
            r = C.run_season(O, "milp_hybrid", models, seed, eps=e); r.update(rep=rep, season=i, regime=O["regime"], family="hybrid", level=e); fr.append(r)
        print(f"frontier rep {rep} season {i} done", flush=True)
        json.dump(fr, open(f"{OUT}/frontier.json", "w"))
print("frontier done")

## 10. Where and when sorties are flown, and sensitivity to the pollen budget

In [ ]:
import json, numpy as np
import sys
C = sys.modules['__main__']
OUT = C.OUT; ex = dict(log=[], budget=[])
rep = 0; rng = np.random.default_rng(500 + rep)
train = [C.orchard(np.random.default_rng(1000 * rep + i), "nominal") for i in range(8)]
test = [C.orchard(np.random.default_rng(1000 * rep + 100 + i), r) for i, r in enumerate(C.REGIMES_TEST)]
T, RH, W, S, Rn, hod, K, Nn = C.trial_data(train, rng)
models = dict(pcf=C.PCF().fit(T, RH, W, Rn, K, Nn))
for i, O in enumerate(test):
    for pol in ["calendar", "greedy", "milp_mean", "milp_scen"]:
        r = C.run_season(O, pol, models, 9000 + i, log=True)
        ex["log"].append(dict(policy=pol, regime=O["regime"], season=i, log=r["log"]))
    for pd in [25.0, 60.0]:
        for pol in ["greedy", "milp_scen"]:
            r = C.run_season(O, pol, models, 9000 + i, pollen_day=pd); r.update(policy=pol, regime=O["regime"], season=i, pollen_day=pd); ex["budget"].append(r)
    print("extra season", i, flush=True)
    json.dump(ex, open(f"{OUT}/extra.json", "w"))
print("extra done")

## 11. Reliability of the ensemble wind-exceedance estimate used by the chance constraint

In [ ]:
import json, numpy as np
import sys
C = sys.modules['__main__']
OUT = C.OUT; rows = []
for rep in range(2 if C.FAST else 5):
    for i, reg in enumerate(C.REGIMES_TEST):
        O = C.orchard(np.random.default_rng(1000 * rep + 100 + i), reg); rng = np.random.default_rng(4242 + 10 * rep + i)
        for day in range(C.DAYS - 1):
            F = C.ensemble_forecast(O, day, rng); h0, Hh = F["h0"], F["H"]; dl = C.daylight_idx(h0, Hh); hours = h0 + dl
            oT, ex, oR = C.sensor_offsets(O, day, rng)
            Wm = F["wind"][:, None, dl] * ex[None, :, None]
            P = (Wm > C.W_MAX).mean(0); Pm = (Wm.mean(0) > C.W_MAX).astype(float)
            truth = (O["blk"]["wind"][:, hours] > C.W_MAX).astype(float)
            lead = np.broadcast_to(dl // 24, P.shape)
            rows.append(np.stack([P.ravel(), Pm.ravel(), truth.ravel(), lead.ravel()], 1))
A = np.concatenate(rows)
bins = np.linspace(0, 1, 9); rel = []
for k in range(8):
    m = (A[:, 0] >= bins[k]) & (A[:, 0] < bins[k + 1] + (k == 7))
    if m.any(): rel.append([float(A[m, 0].mean()), float(A[m, 2].mean()), int(m.sum())])
out = dict(rel=rel, base=float(A[:, 2].mean()),
           admitted_scen=float(A[A[:, 0] <= 0.10, 2].mean()), n_admitted_scen=int((A[:, 0] <= 0.10).sum()),
           admitted_mean=float(A[A[:, 1] == 0, 2].mean()), n_admitted_mean=int((A[:, 1] == 0).sum()),
           admitted_eps0=float(A[A[:, 0] == 0, 2].mean()), n=int(len(A)),
           brier_ens=float(np.mean((A[:, 0] - A[:, 2]) ** 2)), brier_mean=float(np.mean((A[:, 1] - A[:, 2]) ** 2)))
json.dump(out, open(f"{OUT}/windcal.json", "w")); print(out)

## 12. Example forecast trajectories during a heatwave (Fig. 7)

In [ ]:
import json, os, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
os.makedirs("figs", exist_ok=True)
import sys
C = sys.modules['__main__']
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif"], "mathtext.fontset": "stix", "font.size": 7.5, "axes.spines.top": False, "axes.spines.right": False})
rng = np.random.default_rng(500)
train = [C.orchard(np.random.default_rng(i), "nominal") for i in range(8)]
T, RH, W, S, Rn, hod, K, Nn = C.trial_data(train, rng); X = C.feats(T, RH, W, S, Rn, hod)
models = dict(pcf=C.PCF().fit(T, RH, W, Rn, K, Nn), gbt=C.GBT().fit(X, K, Nn))
hi = np.percentile(T, 99.5)
O = C.orchard(np.random.default_rng(102), "heatwave"); rng = np.random.default_rng(11)
ev = np.where(O["st"]["event"] > 3)[0]; d0 = max(0, int(ev[0]) - 1) if len(ev) else 4
tt, tru, pc_, gb_, Tb = [], [], [], [], []
for day in range(d0, min(d0 + 5, C.DAYS - 1)):
    F = C.ensemble_forecast(O, day, rng); h0, Hh = F["h0"], F["H"]; dl = C.daylight_idx(h0, Hh); dl = dl[dl < 24]; hours = h0 + dl
    oT, ex, oR = C.sensor_offsets(O, day, rng); b = 0
    Tm = F["T"][:, dl] + oT[b]; Wm = F["wind"][:, dl] * ex[b]; Rm = np.clip(F["RH"][:, dl] + oR[b], 8, 100); Sm = F["solar"][:, dl]; Rr = F["rain"][:, dl]
    for name, L in [("pcf", pc_), ("gbt", gb_)]:
        L.append(models[name].predict(Tm, Rm, Wm, Sm, Rr, np.broadcast_to(hours % 24, Tm.shape)).mean(0))
    tt.append(hours / 24); tru.append(O["blk"]["succ"][b, hours]); Tb.append(O["blk"]["T"][b, hours])
tt, tru, pc_, gb_, Tb = map(np.concatenate, (tt, tru, pc_, gb_, Tb))
fig, ax = plt.subplots(figsize=(7.16, 1.9))
oor = Tb > hi
for i in range(len(tt)):
    if oor[i]: ax.axvspan(tt[i] - 0.5 / 24, tt[i] + 0.5 / 24, color="0.88", lw=0)
for d in np.unique(np.floor(tt)):
    m = np.floor(tt) == d
    ax.plot(tt[m], tru[m], color="black", lw=1.4, label="truth" if d == np.floor(tt[0]) else None)
    ax.plot(tt[m], pc_[m], color="black", lw=0.9, ls="--", label="PCF day-ahead" if d == np.floor(tt[0]) else None)
    ax.plot(tt[m], gb_[m], color="0.5", lw=0.9, ls="-.", label="GBT day-ahead" if d == np.floor(tt[0]) else None)
ax.set_xlabel("Day of season (daylight hours only; grey bands: temperature above the trial range)", fontsize=7); ax.set_ylabel("Success probability $v$", fontsize=7); ax.set_ylim(0, 1)
ax.legend(fontsize=6, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.savefig("figs/fig10_heatwave_ts.png", dpi=400, bbox_inches="tight", pad_inches=0.03)
json.dump(dict(mae_pcf_oor=float(np.abs(pc_ - tru)[oor].mean()) if oor.any() else None, mae_gbt_oor=float(np.abs(gb_ - tru)[oor].mean()) if oor.any() else None,
               n_oor=int(oor.sum()), days=[int(d0), int(min(d0 + 5, C.DAYS - 1))]), open("results/ts.json", "w"))
print("ok", oor.sum())

## 13. Ensemble size and empirical risk of the chance constraint

In [ ]:
import json, numpy as np, time
import sys
C = sys.modules['__main__']
OUT = C.OUT; res = []
_orig_ens = C.ensemble_forecast
for rep in range(1 if C.FAST else 3):
    rng = np.random.default_rng(500 + rep)
    train = [C.orchard(np.random.default_rng(1000 * rep + i), "nominal") for i in range(8)]
    test = [C.orchard(np.random.default_rng(1000 * rep + 100 + i), r) for i, r in enumerate(C.REGIMES_TEST)]
    T, RH, W, S, Rn, hod, K, Nn = C.trial_data(train, rng)
    models = dict(pcf=C.PCF().fit(T, RH, W, Rn, K, Nn))
    for i, O in enumerate(test):
        seed = 9000 + 100 * rep + i
        for M in [4, 8, 16, 32, 64]:
            C.ensemble_forecast = (lambda O_, d_, r_, m=M: _orig_ens(O_, d_, r_, m))
            t0 = time.time()
            r = C.run_season(O, "milp_scen", models, seed, eps=0.10)
            r.update(rep=rep, season=i, regime=O["regime"], M=M, wall=time.time() - t0); res.append(r)
        C.ensemble_forecast = _orig_ens
        print(f"ensize rep {rep} season {i} done", flush=True)
        json.dump(res, open(f"{OUT}/ensize.json", "w"))
print("ensize done")

## 14. Aggregation of all results, statistics and figures

In [ ]:
"""Aggregates results into numbers.json and grayscale figures."""
import json, os, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
os.makedirs("figs", exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif"], "mathtext.fontset": "stix", "font.size": 7.5,
                     "axes.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
R = json.load(open("results/results.json"))
N = {"n_rep": len(R)}
def ms(v): v = np.array([x for x in v if x is not None], float); return [float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0]
REG = ["nominal", "heatwave", "cold_spell"]; FC = ["pcf", "gbt", "mlp", "clim"]
# ---------- forecast skill ----------
N["skill"] = {}
for reg in REG:
    N["skill"][reg] = {}
    for f in FC:
        rows = [s[f] for r in R for s in r["skill"][reg]]
        N["skill"][reg][f] = {k: ms([x[k] for x in rows]) for k in ["brier", "logloss", "mae", "mae_d1", "mae_d2", "mae_oor"]}
        N["skill"][reg][f]["n_oor"] = int(np.sum([x["n_oor"] for x in rows]))
N["skill_no_sensor"] = {k: ms([r["skill_no_sensor"]["pcf"][k] for r in R]) for k in ["brier", "mae"]}
N["skill_with_sensor_nom0"] = {k: ms([r["skill"]["nominal"][0]["pcf"][k] for r in R]) for k in ["brier", "mae"]}
th = np.array([r["theta"] for r in R]); N["theta"] = [[float(a), float(b)] for a, b in zip(th.mean(0), th.std(0, ddof=1))]
# ---------- policies ----------
POL = ["calendar", "greedy", "milp_mean", "milp_scen", "milp_oracle"]
S = [s for r in R for s in r["seasons"]]
def get(pol, reg=None, key="fruit_set"):
    return np.array([s[key] for s in S if s["policy"] == pol and (reg is None or s["regime"] == reg)], float)
N["pol"] = {}
for reg in REG + [None]:
    kk = reg or "all"; N["pol"][kk] = {}
    for p in POL:
        N["pol"][kk][p] = {k: ms(get(p, reg, k)) for k in ["fruit_set", "pollen_per_fruit", "sorties", "viol_rate", "solve_s", "nat_fert", "art_fert"]}
        N["pol"][kk][p]["gap"] = ms((get("milp_oracle", reg) - get(p, reg)) / get("milp_oracle", reg))
# ---------- ablation ----------
A = [a for r in R for a in r["ablation"]]
N["abl"] = {}
for reg in REG + [None]:
    kk = reg or "all"; N["abl"][kk] = {}
    for tag in sorted(set(a["tag"] for a in A)):
        rows = [a for a in A if a["tag"] == tag and (reg is None or a["regime"] == reg)]
        N["abl"][kk][tag] = {k: ms([a[k] for a in rows]) for k in ["fruit_set", "pollen_per_fruit", "viol_rate", "sorties"]}
# ---------- statistics ----------
def holm(p):
    p = np.array(p); o = np.argsort(p); adj = np.empty_like(p); run = 0; m = len(p)
    for rank, i in enumerate(o): run = max(run, (m - rank) * p[i]); adj[i] = min(1, run)
    return adj
def rbc(d):
    nz = d[d != 0]
    if not len(nz): return 0.0
    rk = np.argsort(np.argsort(np.abs(nz))) + 1; return float((rk[nz > 0].sum() - rk[nz < 0].sum()) / rk.sum())
stats = []
for key, lab in [("fruit_set", "fruit set"), ("viol_rate", "violation rate"), ("pollen_per_fruit", "pollen per fruit")]:
    a = get("milp_scen", None, key)
    for b in ["calendar", "greedy", "milp_mean"]:
        bb = get(b, None, key); d = a - bb
        p = wilcoxon(a, bb).pvalue if np.any(d != 0) else 1.0
        stats.append(dict(metric=lab, base=b, mean_diff=float(d.mean()), p=float(p), rbc=rbc(d), n=int(len(d))))
# forecaster comparison on per-season MAE (all regimes) and out-of-range MAE
for f in ["gbt", "mlp"]:
    a = np.array([s["pcf"]["mae"] for r in R for reg in REG for s in r["skill"][reg]]); b = np.array([s[f]["mae"] for r in R for reg in REG for s in r["skill"][reg]])
    stats.append(dict(metric="forecast MAE", base=f, mean_diff=float((a - b).mean()), p=float(wilcoxon(a, b).pvalue), rbc=rbc(a - b), n=int(len(a))))
    a = np.array([s["pcf"]["mae_oor"] for r in R for reg in ["heatwave", "cold_spell"] for s in r["skill"][reg] if s["pcf"]["mae_oor"] is not None])
    b = np.array([s[f]["mae_oor"] for r in R for reg in ["heatwave", "cold_spell"] for s in r["skill"][reg] if s[f]["mae_oor"] is not None])
    stats.append(dict(metric="out-of-range MAE", base=f, mean_diff=float((a - b).mean()), p=float(wilcoxon(a, b).pvalue), rbc=rbc(a - b), n=int(len(a))))
adj = holm([s["p"] for s in stats])
for s, x in zip(stats, adj): s["p_holm"] = float(x)
N["stats"] = stats
# ---------- frontier ----------
if os.path.exists("results/frontier.json"):
    FR = json.load(open("results/frontier.json"))
    N["frontier"] = {}
    for fam in ["mean+margin", "scenario", "hybrid"]:
        lv = sorted(set(x["level"] for x in FR if x["family"] == fam))
        N["frontier"][fam] = [dict(level=l, fruit_set=ms([x["fruit_set"] for x in FR if x["family"] == fam and x["level"] == l]),
                                   viol=ms([x["viol_rate"] for x in FR if x["family"] == fam and x["level"] == l]),
                                   pollen=ms([x["pollen_per_fruit"] for x in FR if x["family"] == fam and x["level"] == l]),
                                   n=int(sum(1 for x in FR if x["family"] == fam and x["level"] == l))) for l in lv]
json.dump(N, open("numbers.json", "w"), indent=1)

# =================== figures ===================
def save(fig, name): fig.savefig(f"figs/{name}.png", dpi=400, bbox_inches="tight", pad_inches=0.03); plt.close(fig)
LAB = {"pcf": "PCF (proposed)", "gbt": "Gradient-boosted trees", "mlp": "MLP", "clim": "Climatology"}
PL = {"calendar": "Calendar", "greedy": "Threshold", "milp_mean": "Mean-forecast MILP", "milp_scen": "Scenario CC-MILP", "milp_oracle": "Oracle MILP"}

# Fig. 2: example heatwave season trace (block 0)
tr = R[0]["trace"]; h = np.arange(len(tr["T"])) / 24
fig, axs = plt.subplots(3, 1, figsize=(7.16, 3.2), sharex=True, gridspec_kw=dict(hspace=0.15))
axs[0].plot(h, tr["T"], color="black", lw=0.6); axs[0].set_ylabel("$T$ (°C)", fontsize=7)
axs[0].axhline(R[0]["curves"]["train_T"][1], color="0.4", ls=":", lw=0.7); axs[0].text(0.1, R[0]["curves"]["train_T"][1] + 0.6, "99th percentile of trial temperatures", fontsize=6.2, color="0.3")
axs[1].plot(h, tr["wind"], color="black", lw=0.6); axs[1].axhline(4.5, color="0.4", ls=":", lw=0.7); axs[1].set_ylabel("$w$ (m s$^{-1}$)", fontsize=7)
axs[1].text(0.1, 4.7, "$w_{\\max}$", fontsize=6.5, color="0.3")
axs[2].fill_between(h, 0, tr["succ"], color="0.75", lw=0); axs[2].plot(h, tr["succ"], color="black", lw=0.5); axs[2].set_ylabel("true $v$", fontsize=7)
axs[2].set_xlabel("Day of bloom season", fontsize=7); axs[2].set_ylim(0, 1)
for ax in axs: ax.tick_params(labelsize=6.5)
save(fig, "fig2_trace")

# Fig. 3: temperature response curves
c = R[0]["curves"]; T = np.array(c["T"])
fig, ax = plt.subplots(figsize=(3.5, 2.1))
ax.axvspan(c["train_T"][0], c["train_T"][1], color="0.92", lw=0)
ax.text((c["train_T"][0] + c["train_T"][1]) / 2, 0.97, "range of trial temperatures", ha="center", fontsize=6.2, color="0.3")
ax.plot(T, c["truth"], color="black", lw=1.6, label="Simulator truth (unknown to models)")
ax.plot(T, c["pcf"], color="black", lw=0.9, ls="--", label=LAB["pcf"])
ax.plot(T, c["gbt"], color="0.45", lw=0.9, ls="-.", label=LAB["gbt"])
ax.plot(T, c["mlp"], color="0.6", lw=0.9, ls=":", label=LAB["mlp"])
ax.set_xlabel("Air temperature (°C), RH = 60%, $w$ = 1.5 m s$^{-1}$", fontsize=7); ax.set_ylabel("Success probability $v$", fontsize=7); ax.set_ylim(0, 1.05)
ax.legend(fontsize=5.9, loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2)
save(fig, "fig3_response")

# Fig. 4: reliability diagrams pooled
fig, axs = plt.subplots(1, 4, figsize=(7.16, 1.85), gridspec_kw=dict(wspace=0.35))
for ax, f in zip(axs, FC):
    pts = np.array([b for r in R for reg in REG for s in r["skill"][reg] for b in s[f]["calib"]])
    ed = np.linspace(0, 1, 11); xs, ys, ws = [], [], []
    for i in range(10):
        m = (pts[:, 0] >= ed[i]) & (pts[:, 0] < ed[i + 1] + (i == 9))
        if m.any(): w = pts[m, 2]; xs.append(np.average(pts[m, 0], weights=w)); ys.append(np.average(pts[m, 1], weights=w)); ws.append(w.sum())
    ax.plot([0, 1], [0, 1], ls=":", color="0.4", lw=0.7); ax.plot(xs, ys, color="black", marker="o", ms=2.5, lw=0.9)
    ws = np.array(ws); ax.bar(xs, ws / ws.sum() * 0.6, width=0.07, color="0.82", edgecolor="0.5", lw=0.3)
    ax.set_title(LAB[f], fontsize=7); ax.set_xlabel("Predicted $v$", fontsize=7); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
axs[0].set_ylabel("Observed germination", fontsize=7)
save(fig, "fig4_calibration")

# Fig. 5: forecast error by regime incl. out-of-range
fig, axs = plt.subplots(1, 2, figsize=(7.16, 1.9), gridspec_kw=dict(wspace=0.22))
x = np.arange(3); wb = 0.2; cols = ["0.15", "0.55", "0.8", "white"]
for i, f in enumerate(FC):
    m = [N["skill"][reg][f]["mae"][0] for reg in REG]; e = [N["skill"][reg][f]["mae"][1] for reg in REG]
    axs[0].bar(x + (i - 1.5) * wb, m, wb, yerr=e, color=cols[i], edgecolor="black", lw=0.4, capsize=1.2, error_kw=dict(lw=0.5), label=LAB[f])
    m2 = [N["skill"][reg][f]["mae_oor"][0] for reg in ["heatwave", "cold_spell"]]; e2 = [N["skill"][reg][f]["mae_oor"][1] for reg in ["heatwave", "cold_spell"]]
    axs[1].bar(np.arange(2) + (i - 1.5) * wb, m2, wb, yerr=e2, color=cols[i], edgecolor="black", lw=0.4, capsize=1.2, error_kw=dict(lw=0.5))
axs[0].set_xticks(x); axs[0].set_xticklabels(["Nominal", "Heatwave", "Cold spell"]); axs[0].set_ylabel("MAE of $v$ (all hours)", fontsize=7)
axs[1].set_xticks(range(2)); axs[1].set_xticklabels(["Heatwave", "Cold spell"]); axs[1].set_ylabel("MAE of $v$ (out-of-range hours)", fontsize=7)
axs[0].legend(fontsize=6, ncol=4, loc="upper center", bbox_to_anchor=(1.05, 1.22))
save(fig, "fig5_forecast_error")

# Fig. 6: fruit set and violation by policy and regime
fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.0), gridspec_kw=dict(wspace=0.25))
cols = ["white", "0.85", "0.6", "0.15", "0.4"]; hat = ["", "////", "", "", "xxxx"]; wb = 0.16
for i, p in enumerate(POL):
    m = [N["pol"][reg][p]["fruit_set"][0] * 100 for reg in REG]; e = [N["pol"][reg][p]["fruit_set"][1] * 100 for reg in REG]
    axs[0].bar(x + (i - 2) * wb, m, wb, yerr=e, color=cols[i], hatch=hat[i], edgecolor="black", lw=0.4, capsize=1.2, error_kw=dict(lw=0.5), label=PL[p])
    m = [N["pol"][reg][p]["viol_rate"][0] * 100 for reg in REG]; e = [N["pol"][reg][p]["viol_rate"][1] * 100 for reg in REG]
    axs[1].bar(x + (i - 2) * wb, m, wb, yerr=e, color=cols[i], hatch=hat[i], edgecolor="black", lw=0.4, capsize=1.2, error_kw=dict(lw=0.5))
for ax in axs: ax.set_xticks(x); ax.set_xticklabels(["Nominal", "Heatwave", "Cold spell"])
axs[1].set_ylim(bottom=0); axs[0].set_ylabel("Fruit set (%)", fontsize=7); axs[1].set_ylabel("Sorties above $w_{\\max}$ (%)", fontsize=7)
axs[0].set_ylim(30, 85)
axs[0].legend(fontsize=6, ncol=5, loc="upper center", bbox_to_anchor=(1.1, 1.22))
save(fig, "fig6_policies")

# Fig. 7: risk-return frontier
if "frontier" in N:
    fig, ax = plt.subplots(figsize=(3.5, 2.6))
    OFF = {("mean+margin", 0.0): (0, 9), ("mean+margin", 0.5): (12, 10), ("mean+margin", 1.0): (-14, -12), ("mean+margin", 1.5): (12, 0),
           ("scenario", 0.05): (12, 0), ("scenario", 0.1): (16, -16), ("scenario", 0.2): (0, 11), ("scenario", 0.35): (0, -10), ("scenario", 0.5): (0, -10),
           ("hybrid", 0.05): (-16, 8), ("hybrid", 0.1): (16, -10), ("hybrid", 0.2): (-14, 10), ("hybrid", 0.35): (10, -12), ("hybrid", 0.5): (-4, -11)}
    sty = {"mean+margin": ("0.5", "--", "s", "Mean MILP, wind margin $\\delta$"), "scenario": ("black", "-", "o", "Scenario CC-MILP, risk $\\varepsilon$"), "hybrid": ("0.3", ":", "^", "Mean objective + CC, $\\varepsilon$")}
    for fam, (col, ls, mk, lab) in sty.items():
        pts = N["frontier"][fam]
        xs = [p["viol"][0] * 100 for p in pts]; ys = [p["fruit_set"][0] * 100 for p in pts]
        ax.plot(xs, ys, color=col, ls=ls, marker=mk, ms=3.2, lw=0.9, mfc="white" if fam != "scenario" else "black", label=lab)
        for p, x_, y_ in zip(pts, xs, ys):
            off = OFF.get((fam, round(p["level"], 2)), (4, -8))
            ax.annotate(f"{p['level']:g}", (x_, y_), textcoords="offset points", xytext=off, fontsize=5.4, color="0.15",
                        ha="center", va="center", arrowprops=dict(arrowstyle="-", lw=0.35, color="0.45", shrinkA=0, shrinkB=2.2))
    ax.set_xlabel("Sorties flown above $w_{\\max}$ (%)", fontsize=7); ax.set_ylabel("Fruit set (%)", fontsize=7)
    ax.legend(fontsize=5.9, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=1)
    save(fig, "fig7_frontier")
print("done", len(R))

# ---------- wind-exceedance reliability, sortie allocation and pollen budget ----------
if os.path.exists("results/windcal.json"):
    WC = json.load(open("results/windcal.json")); N["windcal"] = WC
    EX = json.load(open("results/extra.json")) if os.path.exists("results/extra.json") else None
    fig, axs = plt.subplots(1, 3, figsize=(7.16, 2.0), gridspec_kw=dict(wspace=0.42))
    rel = np.array(WC["rel"])
    axs[0].plot([0, 1], [0, 1], ls=":", color="0.4", lw=0.7); axs[0].plot(rel[:, 0], rel[:, 1], color="black", marker="o", ms=2.6, lw=0.9)
    axs[0].set_xlabel("Ensemble exceedance $\\hat{P}_{b,h}$", fontsize=7); axs[0].set_ylabel("Observed exceedance", fontsize=7); axs[0].set_title("(a) reliability of $\\hat{P}$", fontsize=7.2)
    if EX:
        PL2 = {"calendar": "Calendar", "greedy": "Threshold", "milp_mean": "Mean MILP", "milp_scen": "Scenario CC-MILP"}
        sty = {"calendar": ("0.65", ":"), "greedy": ("0.45", "-."), "milp_mean": ("0.3", "--"), "milp_scen": ("black", "-")}
        N["alloc"] = {}
        for p in PL2:
            L = [row for e in EX["log"] if e["policy"] == p for row in e["log"]]
            L = np.array(L) if L else np.zeros((0, 5))
            hist = np.zeros(24)
            for hh, s_, T_, w_, n_ in L: hist[int(hh)] += n_
            hist = hist / max(1, hist.sum())
            axs[1].plot(range(6, 20), hist[6:20] * 100, color=sty[p][0], ls=sty[p][1], lw=1.0, label=PL2[p])
            ws = L[:, 4] if len(L) else np.ones(1)
            N["alloc"][p] = dict(mean_success=float(np.average(L[:, 1], weights=ws)) if len(L) else None, mean_T=float(np.average(L[:, 2], weights=ws)) if len(L) else None,
                                 share_low=float(np.sum(ws[L[:, 1] < 0.3]) / ws.sum()) if len(L) else None)
        axs[1].set_xlabel("Hour of day", fontsize=7); axs[1].set_ylabel("Share of sorties (%)", fontsize=7); axs[1].set_title("(b) when sorties are flown", fontsize=7.2)
        axs[1].legend(fontsize=5.6, loc="upper right")
        for i, p in enumerate(PL2):
            L = np.array([row for e in EX["log"] if e["policy"] == p for row in e["log"]])
            vals = np.repeat(L[:, 1], L[:, 4].astype(int))
            bp = axs[2].boxplot([vals], positions=[i], widths=0.55, patch_artist=True, showfliers=False, medianprops=dict(color="black", lw=0.9))
            for b_ in bp["boxes"]: b_.set_facecolor(["white", "0.85", "0.6", "0.3"][i]); b_.set_linewidth(0.6)
        axs[2].set_xticks(range(4)); axs[2].set_xticklabels(["Cal.", "Thr.", "Mean", "CC"], fontsize=6.5)
        axs[2].set_ylabel("True $v$ at sortie time", fontsize=7); axs[2].set_title("(c) realised success of sorties", fontsize=7.2)
        B = EX["budget"]; N["budget"] = {}
        for pd in [25.0, 60.0]:
            for p in ["greedy", "milp_scen"]:
                rows = [b_ for b_ in B if b_["pollen_day"] == pd and b_["policy"] == p]
                if rows: N["budget"][f"{p}|{pd:g}"] = {k: ms([r[k] for r in rows]) for k in ["fruit_set", "pollen_per_fruit", "sorties", "viol_rate"]}
        for p in ["greedy", "milp_scen"]:
            rows = [s for s in S if s["policy"] == p and s in [x for r in R[:1] for x in r["seasons"]]]
            N["budget"][f"{p}|40"] = {k: ms([r[k] for r in rows]) for k in ["fruit_set", "pollen_per_fruit", "sorties", "viol_rate"]}
    save(fig, "fig8_allocation")
    json.dump(N, open("numbers.json", "w"), indent=1)
    print("allocation figure done")

# ---------- Brier decomposition (Murphy) from binned forecasts ----------
N["murphy"] = {}
for reg in REG:
    N["murphy"][reg] = {}
    for fc in FC:
        pts = np.array([b for r in R for s in r["skill"][reg] for b in s[fc]["calib"]])
        n = pts[:, 2]; o_bar = np.average(pts[:, 1], weights=n)
        rel_ = float(np.sum(n * (pts[:, 0] - pts[:, 1]) ** 2) / n.sum()); res_ = float(np.sum(n * (pts[:, 1] - o_bar) ** 2) / n.sum())
        N["murphy"][reg][fc] = dict(rel=rel_, res=res_, unc=float(o_bar * (1 - o_bar)))
# ---------- response factors: fitted vs simulator ----------
th = np.array(N["theta"])[:, 0]
fig, axs = plt.subplots(1, 3, figsize=(7.16, 1.85), gridspec_kw=dict(wspace=0.38))
D = np.linspace(0, 4, 200)
for T_, ls in [(20, "-"), (30, "--")]:
    axs[0].plot(D, np.exp(-np.maximum(0, D - 0.9) * (0.55 + 0.03 * max(T_ - 20, 0))), color="0.55", ls=ls, lw=1.0, label=f"simulator, {T_} °C")
axs[0].plot(D, np.exp(-th[3] * np.maximum(0, D - th[4])), color="black", lw=1.2, label="PCF fit")
axs[0].set_xlabel("Vapour-pressure deficit $D$ (kPa)", fontsize=7); axs[0].set_ylabel("$g_V$", fontsize=7); axs[0].legend(fontsize=5.6, loc="lower left")
w = np.linspace(0, 8, 200)
axs[1].plot(w, 1 / (1 + np.exp((w - 4.2) / 0.55)), color="0.55", lw=1.0, label="simulator")
axs[1].plot(w, 1 / (1 + np.exp((w - th[5]) / th[6])), color="black", lw=1.2, ls="--", label="PCF fit")
axs[1].axvline(4.5, color="0.4", ls=":", lw=0.7); axs[1].text(4.6, 0.85, "$w_{\\max}$", fontsize=6.3)
axs[1].set_xlabel("Wind speed $w$ (m s$^{-1}$)", fontsize=7); axs[1].set_ylabel("$g_W$", fontsize=7); axs[1].legend(fontsize=5.6, loc="lower left")
# per-season paired comparison
for k, (b_, mk) in enumerate([("calendar", "o"), ("greedy", "s"), ("milp_mean", "^")]):
    a_ = get("milp_scen", None, "fruit_set") * 100; bb = get(b_, None, "fruit_set") * 100
    axs[2].scatter(bb, a_, s=9, marker=mk, facecolor="white" if k == 0 else ("0.6" if k == 1 else "black"), edgecolor="black", lw=0.5, label={"calendar": "vs calendar", "greedy": "vs threshold", "milp_mean": "vs mean MILP"}[b_])
lo_, hi_ = 30, 85
axs[2].plot([lo_, hi_], [lo_, hi_], color="0.4", ls=":", lw=0.7); axs[2].set_xlim(lo_, hi_); axs[2].set_ylim(lo_, hi_)
axs[2].set_xlabel("Comparator fruit set (%)", fontsize=7); axs[2].set_ylabel("Scenario CC-MILP fruit set (%)", fontsize=7); axs[2].legend(fontsize=5.6, loc="lower right")
axs[0].set_title("(a) desiccation factor", fontsize=7.2); axs[1].set_title("(b) deposition factor", fontsize=7.2); axs[2].set_title("(c) season-level pairs", fontsize=7.2)
save(fig, "fig9_factors_pairs")
json.dump(N, open("numbers.json", "w"), indent=1)
print("murphy and factors done")

## 15. Ensemble-size analysis and figure

In [ ]:
import json, os, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif"], "mathtext.fontset": "stix", "font.size": 7.5,
                     "axes.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
E = json.load(open("results/ensize.json"))
Ms = sorted(set(e["M"] for e in E))
def ms(v): v = np.array(v, float); return [float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0]
out = {}
for M in Ms:
    rows = [e for e in E if e["M"] == M]
    out[str(M)] = {k: ms([r[k] for r in rows]) for k in ["fruit_set", "viol_rate", "solve_s", "sorties", "pollen_per_fruit"]}
    out[str(M)]["n"] = len(rows); out[str(M)]["kmax"] = int(np.floor(0.10 * M + 1e-9))
# paired differences against M=16 for each season
base = {(e["rep"], e["season"]): e for e in E if e["M"] == 16}
for M in Ms:
    d = [e["fruit_set"] - base[(e["rep"], e["season"])]["fruit_set"] for e in E if e["M"] == M and (e["rep"], e["season"]) in base]
    out[str(M)]["d_fs_vs16"] = ms(d)
N = json.load(open("numbers.json")); N["ensize"] = out; json.dump(N, open("numbers.json", "w"), indent=1)

fig, ax = plt.subplots(1, 3, figsize=(7.0, 2.05), gridspec_kw=dict(wspace=0.45))
x = np.arange(len(Ms)); se = lambda key, M: out[str(M)][key][1] / np.sqrt(out[str(M)]["n"])
def bar(a, key, scale, ylab, fmt):
    m = np.array([out[str(M)][key][0] for M in Ms]) * scale; s = np.array([se(key, M) for M in Ms]) * scale
    a.errorbar(x, m, yerr=s, fmt=fmt, color="k", ms=4, lw=0.8, capsize=2.5, mfc="white", zorder=3)
    a.plot(x, m, color="k", lw=0.6, ls="-")
    a.set_xticks(x); a.set_xticklabels([str(M) for M in Ms]); a.set_xlabel("Ensemble members $M$"); a.set_ylabel(ylab)
    return m, s
m, s = bar(ax[0], "viol_rate", 100, "Sorties above $w_{\\max}$ (%)", "o")
for i, M in enumerate(Ms):
    ax[0].text(x[i] + 0.12, m[i] + 0.05, f"$k$={out[str(M)]['kmax']}", ha="left", va="bottom", fontsize=6.3)
ax[0].set_ylim(0, max(m + s) * 1.15); ax[0].set_xlim(-0.4, len(Ms) - 0.4)
bar(ax[1], "d_fs_vs16", 100, "Fruit set relative to $M$=16 (points)", "s")
ax[1].axhline(0, color="k", lw=0.5, ls=":")
bar(ax[2], "solve_s", 1, "MILP solve time per plan (s)", "^")
for a, t in zip(ax, "abc"):
    a.text(-0.30, 1.04, f"({t})", transform=a.transAxes, fontsize=8, fontweight="bold")
fig.savefig("figs/fig11_ensemble.png", dpi=400, bbox_inches="tight", pad_inches=0.03); plt.close(fig)
print("ensize fig ok")

## 16. System architecture figure (Fig. 1)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import os
os.makedirs("figs", exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif"], "mathtext.fontset": "stix", "font.size": 6.5})
fig, ax = plt.subplots(figsize=(7.16, 3.6))
ax.set_xlim(0, 104); ax.set_ylim(-1, 53); ax.axis("off")
BOX = {}
g = "#E6E6E6"
def box(key, x, y, w, h, title, sub=None, fill="white", lw=0.8):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.2,rounding_size=0.8", fc=fill, ec="black", lw=lw))
    ax.text(x + w / 2, y + h - 1.3, title, ha="center", va="top", fontsize=6.6, fontweight="bold", linespacing=1.1)
    if sub:
        nl = title.count("\n")
        ax.text(x + w / 2, y + h - 5.0 - 2.6 * nl, sub, ha="center", va="top", fontsize=6.0, linespacing=1.3)
    BOX[key] = (x, y, w, h)
def arrow(p0, p1, ls="-", lw=0.8):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=7, lw=lw, ls=ls, color="black", shrinkA=0, shrinkB=0))
def R(k): x, y, w, h = BOX[k]; return (x + w, y + h / 2)
def L(k): x, y, w, h = BOX[k]; return (x, y + h / 2)
def T(k, f=0.5): x, y, w, h = BOX[k]; return (x + w * f, y + h)
def B(k, f=0.5): x, y, w, h = BOX[k]; return (x + w * f, y)

# inputs column
box("ens", 0.5, 36, 17, 12, "Station ensemble\nforecast", "$T^{(m)}_h$, $RH^{(m)}_h$, $w^{(m)}_h$,\nrain, radiation; 48 h")
box("sens", 0.5, 20, 17, 12, "In-orchard sensors", "block history of\n$T$, $RH$, $w$ (72 h)")
box("phen", 0.5, 4, 17, 12, "Phenology state", "thermal time $\\Theta_b$,\nreceptive pool $R_{b,d}$")
# processing
box("down", 22.5, 28, 17, 12, "Microclimate\ndownscaling", "offsets $\\Delta\\hat{T}_b$, $\\Delta\\widehat{RH}_b$,\nexposure $\\hat{e}_b$")
box("pcf", 44.5, 28, 18, 12, "Physiologically\nconstrained forecaster", "$v=\\pi_0\\, g_T\\, g_V\\, g_W\\,(1-r)$", fill=g)
box("trial", 43.5, 44.0, 20, 7.0, "Viability trials", "binomial germination counts")
box("scen", 67, 28, 15.5, 12, "Scenario tensor", "success $v_{b,h}^{(m)}$\n\nwind risk $\\hat{P}_{b,h}$")
box("milp", 67, 4.5, 15.5, 17.5, "Chance-constrained\nMILP", "max scenario-mean\nfertilised flowers s.t.\nfleet, battery, pollen,\n$\\hat{P}_{b,h}\\leq\\varepsilon$", fill=g)
box("exec", 87.5, 4.5, 15.5, 17.5, "Execute day 1", "sorties $x_{b,h}$ flown;\nnew observations,\nupdated pools")

arrow(R("ens"), (22.5, 37.0))
arrow(R("sens"), (22.5, 31.0))
arrow(R("down"), L("pcf"))
arrow(R("pcf"), L("scen"))
arrow(B("trial"), T("pcf"), ls="--")
ax.text(B("trial")[0] - 0.7, 42.1, "maximum-likelihood fit", fontsize=5.6, style="italic", ha="right", va="center")
arrow(B("scen"), T("milp"))
ax.text(B("scen")[0] + 0.7, 24.5, "$M$ scenarios", fontsize=5.8, style="italic", ha="left", va="center")
arrow(R("phen"), (67, 10.0))
ax.text(42.0, 10.8, "receptive pools $R_{b,d}$", fontsize=5.8, style="italic", ha="center")
arrow(R("milp"), L("exec"))
# rolling-horizon feedback: exec -> phenology and sensors (loop below boxes)
ax.plot([95.25, 95.25, 9.0], [4.5, 1.2, 1.2], color="black", lw=0.8)
arrow((9.0, 1.2), (9.0, 3.8))
ax.text(52, 1.7, "rolling horizon: next morning, re-plan with new forecast, sensor data and pools", fontsize=5.8, style="italic", ha="center", va="bottom")
# legend
ax.plot([71, 74], [49.5, 49.5], color="black", lw=0.8); ax.text(74.6, 49.5, "decision flow", va="center", fontsize=5.8)
ax.plot([71, 74], [46.8, 46.8], color="black", lw=0.8, ls="--"); ax.text(74.6, 46.8, "offline model fitting", va="center", fontsize=5.8)
ax.add_patch(FancyBboxPatch((71.2, 43.3), 2.6, 1.6, boxstyle="round,pad=0.1", fc=g, ec="black", lw=0.5)); ax.text(74.6, 44.1, "core contribution", va="center", fontsize=5.8)
fig.savefig("figs/fig1_architecture.png", dpi=400, bbox_inches="tight", pad_inches=0.03)
print("ok")